In [ ]:
import pandas as pd
import numpy as np
import rostpy
import matplotlib.pyplot as plt
from pathlib import Path

# Locate the repository root so file paths work from the repo root or this notebook folder.
REPO_DIR = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists())

# Use the Python process pipeline outputs: raw IFCB exports -> clean carbon + taxonomy tables.
DATA_DIR = REPO_DIR / "data" / "NESLTER_transect"

def taxon_mapping(
    df_full: pd.DataFrame,
    taxonomy: pd.DataFrame,
    from_level: str = 'Annotations',
    to_level: str = 'Genus',
    aggfunc = 'sum'
) -> tuple[pd.DataFrame, list]:
    """
    Aggregates concentration or count data in `df_full` from one taxonomic level to another,
    while preserving metadata columns and returning new taxon column names.
    Parameters:
        df_full (pd.DataFrame): Input DataFrame where some columns are taxa at `from_level`.
        taxonomy (pd.DataFrame): DataFrame mapping taxonomic levels (must contain `from_level` and `to_level`).
        from_level (str): Taxonomic level currently used in df_full columns.
        to_level (str): Taxonomic level to which data should be aggregated.
        aggfunc (str or function): Aggregation method ('sum', 'mean', etc.), passed to .groupby().
    Returns:
        Tuple:
            - pd.DataFrame: DataFrame with metadata columns + aggregated taxonomic columns.
            - list: List of new taxon column names (aggregated `to_level` values).
    """
    if from_level not in taxonomy.columns or to_level not in taxonomy.columns:
        raise ValueError(f"Taxonomy table must include columns '{from_level}' and '{to_level}'")
    # Strip whitespace from taxonomy DataFrame
    taxonomy = taxonomy.apply(lambda col: col.map(lambda x: x.strip() if isinstance(x, str) else x))
    # Build mapping dictionary from taxonomy
    mapping_dict = taxonomy.set_index(from_level)[to_level].dropna().to_dict()
    # Taxon columns that can be mapped
    mapped_taxa = [col for col in df_full.columns if col in mapping_dict]
    # Columns not in the mapping (assumed to be metadata)
    unmapped_cols = [col for col in df_full.columns if col not in mapping_dict]
    # Rename and group mapped taxon columns
    df_full_mapped = df_full[mapped_taxa].copy()
    df_full_mapped.columns = [mapping_dict[col] for col in df_full_mapped.columns]
    # Aggregate mapped data
    df_full_agg = df_full_mapped.T.groupby(level=0).agg(aggfunc).T
    # Concatenate metadata + aggregated taxa
    df_full_result = pd.concat([df_full[unmapped_cols], df_full_agg], axis=1)
    # Return both the result and the new taxon column names
    return df_full_result, sorted(df_full_agg.columns.tolist())

In [ ]:
df = pd.read_csv(DATA_DIR / "ifcb_carbon_clean.csv")
taxonomy = pd.read_csv(DATA_DIR / "ifcb_taxonomy.csv")
df, taxa_cols = taxon_mapping(
    df,
    taxonomy=taxonomy,
    from_level="Annotations",
    to_level="Mix_Taxa",
    aggfunc="sum",
)
df = df[(df['cruise']=='HRS2601') & (df['sample_type']=='underway')].reset_index(drop=True)
df['sample_time'] = pd.to_datetime(df['sample_time'])

In [ ]:
from matplotlib import ticker
fig, ax = plt.subplots(figsize=(10, 5))
var = 'Lauderia annulata'
sc = ax.scatter(
    df['sample_time'],
    df['latitude'],
    c=df[var],
    cmap='jet',
    s=20
)
ax.set_xlabel('Sample Time')
ax.set_ylabel('Latitude')
cbar = plt.colorbar(sc, ax=ax, format=ticker.ScalarFormatter(useMathText=True))
cbar.set_label(f'{var.capitalize()} (µg C L$^{{-1}}$)', fontsize=10)
formatter = ticker.ScalarFormatter(useMathText=True)
formatter.set_scientific(True)
formatter.set_powerlimits((0, 0))  # always show ×10^n
cbar.formatter = formatter
cbar.update_ticks()
ax.grid()
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(11, 8),
    gridspec_kw={"height_ratios": [1, 1]},
)
sc = ax1.scatter(
    df["sample_time"],
    df["latitude"],
    c=df['temperature'],
    cmap="turbo",
    edgecolor="gray",
    linewidth=0.5,
    vmin=df['temperature'].quantile(0.05),
    vmax=df['temperature'].quantile(0.95),
)
ax1.set_ylabel("Latitude")
# Inset colorbar
cax = ax1.inset_axes([0.72, 0.1, 0.22, 0.035])
cbar = plt.colorbar(sc, cax=cax, orientation="horizontal")
cbar.set_label(
    f"Temperature (°C)",
    fontsize=10,
    labelpad=3,
)
cbar.ax.xaxis.set_label_position("top")
cbar.ax.tick_params(labelsize=8)
sc = ax2.scatter(
    df["sample_time"],
    df["latitude"],
    c=df['salinity'],
    cmap="turbo",
    edgecolor="gray",
    linewidth=0.5,
    vmin=df['salinity'].quantile(0.05),
    vmax=df['salinity'].quantile(0.95),
)
ax2.set_xlabel("Sample Time")
ax2.set_ylabel("Latitude")
# Inset colorbar
cax2 = ax2.inset_axes([0.72, 0.1, 0.22, 0.035])
cbar2 = plt.colorbar(sc, cax=cax2, orientation="horizontal")
cbar2.set_label(
    f"Salinity (PSU)",
    fontsize=10,
    labelpad=3,
)
cbar2.ax.xaxis.set_label_position("top")
cbar2.ax.tick_params(labelsize=8)    

In [ ]:
# Select the taxa abundance/count columns from the full dataframe.
dataset = df[taxa_cols]

# Downscale counts by 100,000 and convert to integers because ROST expects
# discrete observation counts rather than continuous abundances.
ds = dataset.div(10000).astype(int)

# Store taxon names and sample/time identifiers.
taxa = dataset.columns
times = df["pid"]

# Initialize a ROST topic model with one word per taxon and K latent communities.
rost = rostpy.ROST_t(
    V=len(taxa),   # vocabulary size: number of taxa
    K=2,           # number of latent communities/topics
    alpha=0.5,     # topic prior
    beta=0.5,      # word/taxon prior
    gamma=0.1,     # spatial/temporal smoothing parameter
)

# Add each sample as an observation.
# Each taxon index is repeated according to its downscaled count.
for t, (_, data) in enumerate(ds.iterrows()):
    obs = []
    for taxon, count in enumerate(data):
        obs.extend([taxon] * count)

    # Use the row index t as the pose/time coordinate for this observation.
    rost.add_observation([t], obs)

# Refine the ROST model using parallel Gibbs-style updates.
for epoch in range(100):
    rostpy.parallel_refine(rost, 15)

# Count how often each latent community is assigned to each sample/time point.
topics = {f"community_{i+1}": [0 for _ in times] for i in range(rost.K)}

for t in range(len(times)):
    for topic in rost.get_topics_for_pose((t,)):
        topics[f"community_{topic+1}"][t] += 1

# Convert community assignment counts into a dataframe indexed by sample/time ID.
topic_df = pd.DataFrame(topics, index=times)

# Normalize community counts within each sample to get community probabilities.
topic_prob_df = topic_df.div(topic_df.sum(axis=1), axis=0)

# Extract the learned taxon distributions for each latent community.
word_topic_matrix = rost.get_topic_model()

wt_matrix_df = pd.DataFrame(
    word_topic_matrix,
    columns=taxa,
    index=[f"community_{i+1}" for i in range(rost.K)]
)

# Normalize taxon probabilities within each community.
wt_matrix_df_norm = wt_matrix_df.div(wt_matrix_df.sum(axis=1), axis=0)

# Reconstruct expected taxon probabilities for each sample by combining:
# sample-level community probabilities × community-level taxon probabilities.
word_probs = topic_prob_df.to_numpy() @ wt_matrix_df_norm.to_numpy()

# Store reconstructed taxon probabilities in a dataframe indexed by sample/time ID.
word_prob_df = pd.DataFrame(word_probs, columns=taxa, index=times)

data = topic_prob_df.reset_index()
data['pid'] = df.pid
data_merged = pd.merge(df, data, on='pid', how='left')
data_merged.sort_values('sample_time', inplace=True)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
data_merged["sample_time"] = pd.to_datetime(data_merged["sample_time"])
community_cols = [
    c for c in data_merged.columns
    if c.startswith("community_")
]
for community in community_cols:
    fig, (ax1, ax2) = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=(11, 8),
        gridspec_kw={"height_ratios": [2, 1]},
    )
    # Top: latitude vs time
    sc = ax1.scatter(
        data_merged["sample_time"],
        data_merged["latitude"],
        c=data_merged[community],
        cmap="Reds",
        edgecolor="gray",
        linewidth=0.5,
        vmin=0,
        vmax=1,
    )
    ax1.set_ylabel("Latitude")
    # Inset colorbar
    cax = ax1.inset_axes([0.72, 0.1, 0.22, 0.035])
    cbar = plt.colorbar(sc, cax=cax, orientation="horizontal")
    cbar.set_label(
        f"{community.replace('_', ' ').capitalize()} Probability",
        fontsize=10,
        labelpad=3,
    )
    cbar.ax.xaxis.set_label_position("top")
    cbar.ax.tick_params(labelsize=8)
    # Bottom: taxa bar plot
    top_taxa = (
        wt_matrix_df_norm.T[community]
        .sort_values(ascending=False)
        .head(10)
    )
    y = range(len(top_taxa))
    ax2.barh(
        y,
        top_taxa.values,
        color="gray",   # replace with bar_colors if you have taxon_colors
    )
    ax2.invert_yaxis()
    for k, (taxon, v) in enumerate(top_taxa.items()):
        ax2.text(
            v - 0.05 if v > 0.8 else v + 0.01,
            k,
            taxon,
            va="center",
            ha="right" if v > 0.8 else "left",
            fontsize=10,
            clip_on=True,
        )
    ax2.set_xlim(0, 1)
    ax2.set_xlabel("Taxa Probability")
    ax2.set_ylabel("")
    ax2.set_yticks(y)
    ax2.tick_params(
        axis="y",
        left=False,
        labelleft=False,
    )
    fig.subplots_adjust(
        left=0.10,
        right=0.95,
        top=0.95,
        bottom=0.10,
        hspace=0.30,
    )
    plt.show()